# Direct Fitting From One CPMG Revival

This notebook assumes:

- you already isolated one revival window of the CPMG signal,
- you know which revival index `k` that window corresponds to,
- you want a direct estimate of `(A, B)` from the Gaussian peak centers and widths.

The workflow is:

1. fit the segment with a sum of Gaussians,
2. convert each Gaussian center and width into `(A, B)` using Eqs. (4) and (6),
3. optionally refine those values with the exact CPMG model.

In [ ]:
from pathlib import Path

from direct_fit import (
    extract_revival_segment,
    fit_spins_from_segment,
    gaussian_component_curves,
    load_cpmg_csv,
)

DATA_PATH = Path("cpmg_data_spinset2_525G_simulated.csv")
B0_GAUSS = 525.0
N_PULSES = 32
REVIVAL_INDEX = 6

tau_us, signal = load_cpmg_csv(DATA_PATH)
segment_tau, segment_signal = extract_revival_segment(
    tau_us,
    signal,
    revival_index=REVIVAL_INDEX,
    b0_gauss=B0_GAUSS,
)

print(f"Loaded {len(segment_tau)} points for revival k={REVIVAL_INDEX}.")
print(f"tau range: {segment_tau[0]:.4f} us to {segment_tau[-1]:.4f} us")

## Direct `(A, B)` estimate

For your own data, the only required inputs are `segment_tau`, `segment_signal`, `REVIVAL_INDEX`, and `B0_GAUSS`.

The default peak threshold is tuned to avoid over-splitting small shoulders. If your experimental segment is noisier or cleaner, adjust `min_prominence`.

For a single revival window, Eqs. (4) and (6) determine `B` and `|omega_L + A|`. The helper therefore chooses the weak-coupling branch by default and also returns the other possibility as `alternate_a_khz`.

If you already know how many dips you want to represent, pass `n_components=...` to force an exact number of Gaussian peaks.

In [ ]:
direct_result = fit_spins_from_segment(
    segment_tau,
    segment_signal,
    revival_index=REVIVAL_INDEX,
    b0_gauss=B0_GAUSS,
    n_components=None,
    min_prominence=None,
    refine=False,
)

print(f"Gaussian RMSE: {direct_result.gaussian_rmse:.6f}")
for row in direct_result.spin_rows():
    print(
        f"spin {row['spin']}: "
        f"tau={row['peak_tau_us']:.6f} us, "
        f"sigma={row['sigma_us']:.6f} us, "
        f"A={row['a_khz']:.3f} kHz, "
        f"B={row['b_khz']:.3f} kHz"
    )

## Plot the fitted Gaussians

There are two useful views:

- the individual Gaussian dips on top of the baseline,
- the positive dip amplitudes after subtracting the baseline.

The second view is often easier to read when several dips overlap.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print("matplotlib is not installed; skipping the Gaussian plot cell.")
else:
    background, dip_curves, signal_curves = gaussian_component_curves(
        segment_tau,
        baseline=direct_result.baseline,
        slope=direct_result.slope,
        gaussians=direct_result.gaussians,
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(segment_tau, segment_signal, color="black", linewidth=2, label="segment")
    axes[0].plot(segment_tau, background, color="0.5", linestyle="--", label="baseline")
    for index, curve in enumerate(signal_curves, start=1):
        axes[0].plot(segment_tau, curve, linewidth=1.5, label=f"G{index}")
    axes[0].plot(segment_tau, direct_result.gaussian_reconstruction, color="red", linewidth=2, alpha=0.8, label="total Gaussian fit")
    axes[0].set_xlabel("tau (us)")
    axes[0].set_ylabel("P_x")
    axes[0].set_title("Each Gaussian in signal space")
    axes[0].legend(ncol=2, fontsize=8)

    axes[1].plot(segment_tau, background - segment_signal, color="black", linewidth=2, label="measured dip profile")
    for index, dip in enumerate(dip_curves, start=1):
        axes[1].plot(segment_tau, dip, linewidth=1.8, label=f"G{index}")
    axes[1].plot(segment_tau, background - direct_result.gaussian_reconstruction, color="red", linewidth=2, alpha=0.8, label="sum of Gaussians")
    axes[1].set_xlabel("tau (us)")
    axes[1].set_ylabel("dip amplitude")
    axes[1].set_title("Each Gaussian after baseline subtraction")
    axes[1].legend(ncol=2, fontsize=8)

    plt.tight_layout()
    plt.show()

## Optional physical refinement

This step uses the exact single-spin modulation model from `model.py` and starts from the direct estimates above.

Because unresolved or overlapping dips can merge in a narrow segment, this refinement is optional rather than the default.

In [ ]:
refined_result = fit_spins_from_segment(
    segment_tau,
    segment_signal,
    revival_index=REVIVAL_INDEX,
    b0_gauss=B0_GAUSS,
    n_pulses=N_PULSES,
    refine=True,
)

print(f"Physical RMSE: {refined_result.physical_rmse:.6f}")
for row in refined_result.spin_rows():
    print(
        f"spin {row['spin']}: "
        f"A_init={row['a_khz']:.3f} kHz, "
        f"B_init={row['b_khz']:.3f} kHz, "
        f"A_refined={row['refined_a_khz']:.3f} kHz, "
        f"B_refined={row['refined_b_khz']:.3f} kHz"
    )

print(f"readout offset = {refined_result.readout_offset:.6f}")
print(f"readout contrast = {refined_result.readout_contrast:.6f}")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print("matplotlib is not installed; skipping the plot cell.")
else:
    plt.figure(figsize=(9, 4))
    plt.plot(segment_tau, segment_signal, label="segment", linewidth=2)
    plt.plot(segment_tau, direct_result.gaussian_reconstruction, label="Gaussian fit", linestyle="--")
    if refined_result.physical_reconstruction is not None:
        plt.plot(segment_tau, refined_result.physical_reconstruction, label="physical refinement", linestyle=":")
    for gaussian in direct_result.gaussians:
        plt.axvline(gaussian.center_us, color="0.8", linewidth=0.8)
    plt.xlabel("tau (us)")
    plt.ylabel("P_x")
    plt.title(f"CPMG revival k={REVIVAL_INDEX}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Use your own segment

If you already have a segment in memory, replace the loading cell with:

```python
segment_tau = ...
segment_signal = ...
REVIVAL_INDEX = ...
B0_GAUSS = ...

direct_result = fit_spins_from_segment(
    segment_tau,
    segment_signal,
    revival_index=REVIVAL_INDEX,
    b0_gauss=B0_GAUSS,
    n_components=3,
    refine=False,
)
```

If you want the exact-model refinement too, pass `n_pulses=...` and `refine=True`.